## Introduction

Firstly, i have to say that i'm begginer in kaggle, so if you think that this notebook can be improved please comment, it's going to be a pleasure to read.

Well, the idea in this notebook is to use transfer learning, the features is obtained from inceptionV3 and the classfier i opted to use Logistic regression, random forest and XGBoost. Firstly, for reasons of low RAM memory i decide to use part of the images (400 for each class) and then use train_test_split to separate the data. 

##  Setup

In [ ]:
import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from keras.preprocessing import image
from keras.applications.inception_v3 import preprocess_input as preprocess_iv3_input
from keras import applications
from keras.models import Model
import random as rd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, plot_confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt     

In [ ]:
#this is test code
iv3_model = applications.inception_v3.InceptionV3(weights='imagenet', 
                                include_top=False, 
                                input_shape=(224, 224,3))
iv3_model.layers.pop()
output = iv3_model.layers[-1].output
iv3_model = Model(iv3_model.input, output)
iv3_model.trainable = False

### Functions

Here i define two functions to extract the features from inceptionV3. The first function is to preprocess the images and the second to extract the features from the images.

In [ ]:
img_dir='../input/shoe-vs-sandal-vs-boot-dataset-15k-images/Shoe vs Sandal vs Boot Dataset/Boot/'

def process_image2arr(path, img_dims=(224, 224)):
    img = image.load_img(path, target_size=img_dims)
    img_arr = image.img_to_array(img)
    img_arr = np.expand_dims(img_arr, axis=0)
    img_arr = preprocess_iv3_input(img_arr)
    return img_arr

def extract_tl_features(model, image_file_name, image_dir=img_dir):
    
    pr_img = process_image2arr(image_dir+image_file_name) #the function above 
    tl_features = model.predict(pr_img)
    tl_features = np.reshape(tl_features, 2048*5*5)
    return tl_features

## Extracting the features for each class and then concatenating the data

In this part i'm creating a diferent dataframe for each class of shoes. I'm selecting 400 images (randomly obtained) and applying the function previously mentioned, but i choose to write in the variable what type of shoes is in the file path. But the idea here is to form a representative data of these images and i think with this method we can achieve this objective.

### Boots

In [ ]:
img_dir='../input/shoe-vs-sandal-vs-boot-dataset-15k-images/Shoe vs Sandal vs Boot Dataset/Boot/'
img_names=os.listdir('../input/shoe-vs-sandal-vs-boot-dataset-15k-images/Shoe vs Sandal vs Boot Dataset/Boot')

n = len(img_names)
m = np.random.randint(1,n,400)
img_names = pd.DataFrame(img_names,columns = ['img_name'])
img_list = img_names.iloc[m,]

In [ ]:
img_features = dict()
img_names = []

#montar a base de dados com os features das imagens (como s?o varias imagens demora)
for img in img_list['img_name']:
    img_features[img] = extract_tl_features(model=iv3_model, image_file_name=img, image_dir=img_dir)

In [ ]:
data_boot = pd.DataFrame.from_dict(img_features, orient='index')
#Colocar uma vari?vel para ser categ?rica no modelo
data_boot['Type']='Boot' 
data_boot.head()

### Sandals

In [ ]:
img_dir='../input/shoe-vs-sandal-vs-boot-dataset-15k-images/Shoe vs Sandal vs Boot Dataset/Sandal/'
img_names=os.listdir('../input/shoe-vs-sandal-vs-boot-dataset-15k-images/Shoe vs Sandal vs Boot Dataset/Sandal')
n = len(img_names)
m = np.random.randint(1,n,400)
img_names = pd.DataFrame(img_names,columns = ['img_name'])
img_list = img_names.iloc[m,]

In [ ]:
img_features = dict()
img_names = []

#montar a base de dados com os features das imagens (como s?o varias imagens demora)
for img in img_list['img_name']:
    img_features[img] = extract_tl_features(model=iv3_model, image_file_name=img, image_dir=img_dir)

In [ ]:
data_sandal = pd.DataFrame.from_dict(img_features, orient='index')
#Colocar uma vari?vel para ser categ?rica no modelo
data_sandal['Type']='Sandal' 
data_sandal.head()

## Shoe

In [ ]:
img_dir='../input/shoe-vs-sandal-vs-boot-dataset-15k-images/Shoe vs Sandal vs Boot Dataset/Shoe/'
img_names=os.listdir('../input/shoe-vs-sandal-vs-boot-dataset-15k-images/Shoe vs Sandal vs Boot Dataset/Shoe')
n = len(img_names)
m = np.random.randint(1,n,400)
img_names = pd.DataFrame(img_names,columns = ['img_name'])
img_list = img_names.iloc[m,]

In [ ]:
img_features = dict()
img_names = []

#montar a base de dados com os features das imagens (como s?o varias imagens demora)
for img in img_list['img_name']:
    img_features[img] = extract_tl_features(model=iv3_model, image_file_name=img, image_dir=img_dir)

In [ ]:
data_shoe=pd.DataFrame.from_dict(img_features, orient='index')
#Colocar uma vari?vel para ser categ?rica no modelo
data_shoe['Type']='Shoe' 
data_shoe.head()

## Forming a complete dataset

Now that we have the 3 dataframes with the features it's time to concatenate and now we have a problem. I was using random forest and the logistic regression with pandas category type, but to use XGBoost the response variable has to be encoded. So what was my idea, i create two different dataframes, one with the response variable encoded and other with the category type.

In [ ]:
data_features = pd.concat([data_boot,data_sandal,data_shoe])
data_features.Type = data_features.Type.astype('category')
data_features.head()

In [ ]:
# Apply label encoder to each column with categorical data
encoder = LabelEncoder()
cols_encoded = pd.DataFrame(encoder.fit_transform(data_features['Type']))

#indexing
cols_encoded.index = data_features.index

cols_encoded.columns = ['Type_enc']

# Add one-hot encoded columns to numerical features
data_features = pd.concat([data_features, cols_encoded], axis=1)
data_xgb = data_features.drop(columns = 'Type')
data_features = data_features.drop(columns = 'Type_enc')

## Modeling

In this part we're going to separate the dataframe using train_test_split to obtain a subset of data to train the model and the rest for test or validation. To evaluate the performance of each method I am considering the confusion matrix.

In [ ]:
y = data_features.Type
x = data_features.loc[:, data_features.columns != 'Type']

x_train, x_test, y_train, y_test = train_test_split(x ,y,test_size = 0.3,random_state = 0)

### Logistic Regression

In [ ]:
model_log = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=400 )
model_log.fit(x_train, y_train)

In [ ]:
pred = model_log.predict(x_test)
confusion = confusion_matrix(y_test, pred)
acc = accuracy_score(y_test, pred)
print('Accuracy Score:',int(acc*100),'%')

In [ ]:
ax= plt.subplot()
sns.heatmap(confusion, annot=True, fmt='g', cmap = 'Greens', ax=ax); 

ax.set_xlabel('Predicted labels');ax.set_ylabel('True labels'); 
ax.set_title('Confusion Matrix'); 
ax.xaxis.set_ticklabels(['Boot', 'Sandal', 'Shoe']); ax.yaxis.set_ticklabels(['Boot', 'Sandal', 'Shoe']);

### Random Forest

In [ ]:
model_rf = RandomForestClassifier(n_estimators = 400)
model_rf.fit(x_train , y_train)

In [ ]:
pred = model_rf.predict(x_test)
confusion = confusion_matrix(y_test, pred)
acc = accuracy_score(y_test, pred)
print('Accuracy Score:',int(acc*100),'%')

In [ ]:
ax= plt.subplot()
sns.heatmap(confusion, annot=True, fmt='g', cmap = 'Greens', ax=ax); 

ax.set_xlabel('Predicted labels');ax.set_ylabel('True labels'); 
ax.set_title('Confusion Matrix'); 
ax.xaxis.set_ticklabels(['Boot', 'Sandal', 'Shoe']); ax.yaxis.set_ticklabels(['Boot', 'Sandal', 'Shoe']);

### XGBoost

In [ ]:
y = data_xgb.Type_enc
x = data_xgb.loc[:, data_features.columns != 'Type_enc']

x_train, x_test, y_train, y_test = train_test_split(x ,y,test_size = 0.3,random_state = 0)

In [ ]:
model_xgb = xgb.XGBClassifier()
model_xgb.fit(x_train , y_train)

In [ ]:
pred = model_xgb.predict(x_test)
confusion = confusion_matrix(y_test, pred)
acc = accuracy_score(y_test, pred)
print('Accuracy Score:',int(acc*100),'%')

In [ ]:
ax= plt.subplot()
sns.heatmap(confusion, annot=True, fmt='g', cmap = 'Greens', ax=ax); 

ax.set_xlabel('Predicted labels');ax.set_ylabel('True labels'); 
ax.set_title('Confusion Matrix'); 
ax.xaxis.set_ticklabels(['Boot', 'Sandal', 'Shoe']); ax.yaxis.set_ticklabels(['Boot', 'Sandal', 'Shoe']);

## Result

Well, we have obtained a good accuracy for each method but XGBoost have impressed me, it was a really good classification, now i'm going to learn a form to use this in the entire group of images. If you read this far i would like to say thank you very much and if you think that i can improve something in this notebook please comment, it's going to be a good help :D